In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

train = pd.read_parquet(Path("../data/interim/train_a3.parquet"))
FIT_CUTOFF = pd.Timestamp("2014-12-31")
TARGET = "defaulted"
fit = train[train["issue_dt"] <= FIT_CUTOFF].copy()
print(train.shape, fit.shape, round(fit[TARGET].mean(), 3))

(30000, 69) (14373, 69) 0.172


### A4 Step 1: Missingness and imputation

Default rate with vs without each value (loans issued ≤ 2014). No gap differs from noise (e.g. bc_util +4 pts with n = 151, noise ±6 pts), so treatments are based on **why** values are missing:

- **mths_since_last_delinq (46%)**: structural, meaning never delinquent. Flag `ever_delinq`; months kept as NaN (own WoE bin, native in XGBoost). Never-delinquent borrowers default only 1 pt less.
- **emp_title / emp_length**: 685 of 686 missing emp_length also lack emp_title, so one group left employment blank. Becomes an "Unknown" category (A5).
- **bc_util (1.1%)**: likely no bankcard (no column to verify). Missing rate stable across years (1.05% vs 1.09%). Flag + median fill.
- **earliest_cr_dt (300 impossible dates, A1)**: flagged borrowers match normal ones (total_acc 24 vs 24; 54% have a past delinquency, proving earlier credit), so only the date is wrong. Imputed as `credit_hist_months` using the median by total_acc quintile (139 → 210 months), more accurate than one global median.
- **revol_util (4)**: recomputed as balance / limit; 0 where there is no revolving credit (balance = limit = 0).

All fill values are fitted on ≤ 2014 and stored for data_spec.json. Pipeline order: clean → impute → cap → transform.

In [2]:
def missing_vs_default(df, target=TARGET):
    """For each column with gaps: how many are missing, and default rate with vs without the value."""
    rows = []
    for c in df.columns:
        if c.endswith(("_cap", "_log")):          # copies of raw columns made in A3
            continue
        miss = df[c].isna()
        if miss.sum() == 0:
            continue
        rows.append({"column": c,
                     "missing_n": int(miss.sum()),
                     "missing_pct": round(100 * miss.mean(), 1),
                     "default_if_missing": round(df.loc[miss, target].mean(), 3),
                     "default_if_present": round(df.loc[~miss, target].mean(), 3)})
    return pd.DataFrame(rows).sort_values("missing_pct", ascending=False).reset_index(drop=True)

missing_vs_default(fit)

,column,missing_n,missing_pct,default_if_missing,default_if_present
0,mths_since_last_delinq,6665,46.4,0.167,0.177
1,emp_title,824,5.7,0.188,0.171
2,emp_length,686,4.8,0.195,0.171
3,bc_util,151,1.1,0.212,0.172
4,earliest_cr_dt,133,0.9,0.150,0.173
5,title,1,0.0,0.000,0.172
6,revol_util,4,0.0,0.250,0.172


In [3]:
# 1. Are missing emp_title and missing emp_length the same people?
print(pd.crosstab(fit["emp_title"].isna(), fit["emp_length"].isna(),
                  rownames=["title missing"], colnames=["length missing"]))

# 2. Which bankcard columns exist (to test "missing bc_util = no bankcard")?
print([c for c in fit.columns if "bc" in c.lower()])

# 3. Can the 4 missing revol_util be recomputed?
fit.loc[fit["revol_util"].isna(), ["revol_bal", "total_rev_hi_lim"]]

length missing  False  True 
title missing               
False           13548      1
True              139    685
['bc_util', 'bc_util_corrected', 'bc_util_cap']


,revol_bal,total_rev_hi_lim
6429,0.0,0.0
14954,35314.0,34000.0
22524,4359.0,5000.0
29502,0.0,0.0


In [4]:
# History length in months for loans with a valid date
fit["hist_months"] = (fit["issue_dt"] - fit["earliest_cr_dt"]).dt.days / 30.44

# 1. Do the flagged borrowers look like people with a credit history?
cols = [c for c in ["total_acc", "open_acc", "revol_bal", "tot_cur_bal"] if c in fit.columns]
print(fit.groupby("earliest_cr_invalid")[cols].median())
print("had a delinquency before (proves past credit):",
      fit.groupby("earliest_cr_invalid")["mths_since_last_delinq"].apply(lambda s: s.notna().mean()).round(3).to_dict())

# 2. Does history length grow with number of accounts? (basis for a smarter fill)
print(fit.groupby(pd.qcut(fit["total_acc"], 5))["hist_months"].median().round(0))

# 3. Is missing bc_util concentrated in certain years?
print(train.groupby(train["issue_dt"].dt.year)["bc_util"].apply(lambda s: s.isna().sum()))

                     total_acc  open_acc  revol_bal  tot_cur_bal
earliest_cr_invalid                                             
0                         24.0      11.0    12117.0      80343.0
1                         24.0      11.0    11825.0      87285.0
had a delinquency before (proves past credit): {0: 0.536, 1: 0.541}
total_acc
(2.999, 16.0]    139.0
(16.0, 21.0]     167.0
(21.0, 27.0]     183.0
(27.0, 35.0]     199.0
(35.0, 102.0]    210.0
Name: hist_months, dtype: float64
issue_dt
2013     49
2014    102
2015    170
Name: bc_util, dtype: int64


In [5]:
flagged = train[train["earliest_cr_invalid"] == 1]
fico_col = [c for c in train.columns if "fico" in c.lower()][0]   # first FICO column

evidence = pd.DataFrame({
    "has_accounts": flagged["total_acc"] > 0,
    "has_fico": flagged[fico_col].notna(),
    "has_balance": (flagged["revol_bal"] > 0) | (flagged["tot_cur_bal"] > 0),
    "past_delinq": flagged["mths_since_last_delinq"].notna(),
})
print(evidence.sum())                                        # how many rows pass each test
print("rows with NO evidence of credit history:", (~evidence.any(axis=1)).sum())
print("total_acc minimum among flagged:", flagged["total_acc"].min())

has_accounts    300
has_fico        300
has_balance     300
past_delinq     171
dtype: int64
rows with NO evidence of credit history: 0
total_acc minimum among flagged: 3.0


In [8]:
import json
CAPS = json.load(open("../reports/a3_caps.json"))

def fit_imputers(df):
    """Learn fill values from the fit sample only. Returns plain numbers (-> data_spec.json)."""
    hist = (df["issue_dt"] - df["earliest_cr_dt"]).dt.days / 30.44
    _, edges = pd.qcut(df["total_acc"], 5, retbins=True)
    edges[0], edges[-1] = -np.inf, np.inf          # future values outside the training range still get a group
    groups = pd.cut(df["total_acc"], edges)
    return {"total_acc_edges": edges.tolist(),
            "hist_months_by_group": hist.groupby(groups, observed=False).median().round(1).tolist(),
            "bc_util_median": float(df["bc_util"].median())}

def apply_imputation(df, p, caps=CAPS):
    out = df.copy()
    # credit history length: real value where the date is valid, group median where it was impossible
    hist = (out["issue_dt"] - out["earliest_cr_dt"]).dt.days / 30.44
    group = pd.cut(out["total_acc"], p["total_acc_edges"], labels=False)
    fill = np.fmax(group.map(dict(enumerate(p["hist_months_by_group"]))),
    out["mths_since_last_delinq"])
    out["credit_hist_months"] = hist.fillna(fill)
    # revol_util: recompute from balance / limit; 0 when there is no revolving credit
    limit = out["total_rev_hi_lim"]
    recomputed = pd.Series(np.where(limit > 0, out["revol_bal"] / limit * 100, 0.0), index=out.index)
    out["revol_util"] = out["revol_util"].fillna(recomputed.round(1))
    # bc_util: keep the information in a flag, then fill
    out["bc_util_missing"] = out["bc_util"].isna().astype(int)
    out["bc_util"] = out["bc_util"].fillna(p["bc_util_median"])
    # structural missing: no value = never delinquent
    out["ever_delinq"] = out["mths_since_last_delinq"].notna().astype(int)
    # re-apply the A3 caps, since two capped fields just got filled
    for c in ["revol_util", "bc_util"]:
        out[f"{c}_cap"] = out[c].clip(upper=caps[c]["upper"])
    return out

IMPUTE = fit_imputers(fit)
print(IMPUTE)
train_imp = apply_imputation(train, IMPUTE)
print(train_imp[["credit_hist_months", "revol_util", "bc_util", "revol_util_cap", "bc_util_cap"]].isna().sum())
print(train_imp.loc[train_imp["earliest_cr_invalid"] == 1, "credit_hist_months"].describe().round(0))

{'total_acc_edges': [-inf, 16.0, 21.0, 27.0, 35.0, inf], 'hist_months_by_group': [139.0, 167.0, 183.0, 199.5, 210.0], 'bc_util_median': 69.4}
credit_hist_months    0
revol_util            0
bc_util               0
revol_util_cap        0
bc_util_cap           0
dtype: int64
count    300.0
mean     178.0
std       26.0
min      139.0
25%      139.0
50%      183.0
75%      200.0
max      210.0
Name: credit_hist_months, dtype: float64


In [9]:
h = hist_months(train)
contra = (train["mths_since_last_delinq"] > h + 1)       # +1 month tolerance for rounding
print("delinquency older than credit history:", int(contra.sum()))

delinquency older than credit history: 0


In [6]:
def hist_months(df):
    return (df["issue_dt"] - df["earliest_cr_dt"]).dt.days / 30.44

def imputation_mae(df_fit, df_eval, k):
    """MAE (months) of predicting history from total_acc group medians; k = number of groups."""
    h_fit, h_eval = hist_months(df_fit), hist_months(df_eval)
    if k == 1:
        pred = pd.Series(h_fit.median(), index=df_eval.index)
    else:
        _, edges = pd.qcut(df_fit["total_acc"], k, retbins=True, duplicates="drop")
        edges[0], edges[-1] = -np.inf, np.inf
        medians = h_fit.groupby(pd.cut(df_fit["total_acc"], edges), observed=False).median()
        groups = pd.cut(df_eval["total_acc"], edges, labels=False)
        pred = groups.map(dict(enumerate(medians.tolist())))
    known = h_eval.notna()
    return round((pred[known] - h_eval[known]).abs().mean(), 1)

later = train[train["issue_dt"] > FIT_CUTOFF]
pd.Series({k: imputation_mae(fit, later, k) for k in [1, 3, 5, 10, 20]}, name="MAE_months")

1     68.7
3     66.1
5     65.7
10    65.3
20    65.2
Name: MAE_months, dtype: float64

### A4 Step 2: Field reliability

Fields are rated by source: platform-calculated, bureau-reported or self-reported. Two tests (loans issued ≤ 2014; the 899 incomes fixed ×1000 in A1 excluded):

- **Round-number share** (multiples of $1,000): annual_inc 84% (self-reported estimates), revol_bal and tot_cur_bal 0.1% (= chance rate, exact bureau records), loan_amnt 61% (chosen round amounts), total_rev_hi_lim 10.7% (round limits per card).
- **By verification status**: round incomes 85% (Not Verified), 86% (Source Verified), 81% (Verified). Verification does not replace the stated income. Verified loans default more (19.3% vs 12.4%) and are larger (median $16k vs $10k): Lending Club verified selectively, so verification_status is confounded with loan size and risk, not a measure of accuracy.

Ratings: **high** = platform and bureau fields; **medium** = home_ownership (consistent with mortgage balances), dti (built on self-reported income), verification_status (confounded); **low** = annual_inc, emp_title, emp_length.

In [10]:
def round_share(df, cols, unit=1000):
    """Share of non-zero values that are exact multiples of `unit`."""
    out = {}
    for c in cols:
        s = df[c].dropna()
        s = s[s > 0]
        out[c] = round((s % unit == 0).mean(), 3)
    return pd.Series(out, name=f"share_multiple_of_{unit}")

clean_inc = fit[fit["annual_inc_corrected"] == 0]          # exclude our ×1000 fixes
print(round_share(clean_inc, ["annual_inc"]))
print(round_share(fit, ["loan_amnt", "revol_bal", "tot_cur_bal", "total_rev_hi_lim", "tot_hi_cred_lim"]))

annual_inc    0.84
Name: share_multiple_of_1000, dtype: float64
loan_amnt           0.610
revol_bal           0.001
tot_cur_bal         0.001
total_rev_hi_lim    0.107
tot_hi_cred_lim     0.016
Name: share_multiple_of_1000, dtype: float64


In [11]:
def by_verification(df, target=TARGET):
    """Per verification group: size, median income, share of round incomes, default rate."""
    g = df.groupby("verification_status")
    return pd.DataFrame({
        "loans": g.size(),
        "median_income": g["annual_inc"].median(),
        "round_income_share": g["annual_inc"].apply(lambda s: (s % 1000 == 0).mean()).round(3),
        "median_loan": g["loan_amnt"].median(),
        "default_rate": g[target].mean().round(3),
    })

by_verification(clean_inc)

,loans,median_income,round_income_share,median_loan,default_rate
verification_status,,,,,
Not Verified,4136,58000.0,0.854,10000.0,0.124
Source Verified,4783,66000.0,0.855,15000.0,0.190
Verified,5011,67000.0,0.813,16000.0,0.193


In [12]:
clean_inc.groupby([pd.qcut(clean_inc["loan_amnt"], 4), "verification_status"],
                  observed=True)[TARGET].mean().unstack().round(3)

verification_status,Not Verified,Source Verified,Verified
loan_amnt,,,
"(999.999, 8000.0]",0.128,0.164,0.150
"(8000.0, 12887.5]",0.135,0.208,0.195
"(12887.5, 20000.0]",0.104,0.203,0.190
"(20000.0, 35000.0]",0.138,0.180,0.217


### A4 Step 3: Leakage and the input list

**Single-feature AUC scan** (each numeric column alone as a score, loans ≤ 2014): bureau_recovery_flag 0.906, far above every other column (next: int_rate 0.681). Recoveries occur after charge-off, so this confirms the A1 leakage finding. Honest predictors range 0.55–0.68 (term, FICO, recent account openings, dti). Raw, capped and log versions have identical AUC, confirming the A3 treatment preserves ranking.

**Excluded from model inputs** (each with a recorded reason):
- leakage: bureau_recovery_flag (and all servicing data, which was never merged)
- target: defaulted
- old-model output: grade, sub_grade, int_rate. The rate is set from sub_grade, i.e. Lending Club's risk opinion; our model must assess risk independently and is benchmarked against grade. The effect of including them is tested in the Part B ablation.
- identifiers and dates; redundant or replaced columns (title, raw text dates, raw/capped versions superseded by treated fields)
- audit flags: they mark planted errors that do not exist in the holdout
- zip_code: high-cardinality location, proxy risk

Pending: emp_title (A5), addr_state_clean (A7), FICO pair (→ fico_mid).

In [13]:
from sklearn.metrics import roc_auc_score

def single_feature_auc(df, target=TARGET):
    """AUC of each numeric column alone vs the label; > 0.75 on one column is suspicious."""
    rows = []
    for c in df.select_dtypes("number").columns:
        if c == target:
            continue
        s = df[c]
        ok = s.notna()
        if s[ok].nunique() < 2:
            continue
        auc = roc_auc_score(df.loc[ok, target], s[ok])
        rows.append({"column": c, "auc": round(max(auc, 1 - auc), 3), "missing": int((~ok).sum())})
    return pd.DataFrame(rows).sort_values("auc", ascending=False).reset_index(drop=True)

auc_table = single_feature_auc(fit)
auc_table.head(15)

,column,auc,missing
0,bureau_recovery_flag,0.906,0
1,int_rate,0.681,0
2,term_months,0.603,0
3,fico_range_low,0.586,0
4,fico_range_high,0.586,0
5,acc_open_past_24mths,0.581,0
6,dti,0.574,0
7,dti_cap,0.574,0
8,num_tl_op_past_12m,0.566,0
9,bc_util_cap,0.557,151


In [14]:
print(len(train_imp.columns))
print(sorted(train_imp.columns))

72
['acc_open_past_24mths', 'addr_state', 'addr_state_clean', 'annual_inc', 'annual_inc_cap', 'annual_inc_corrected', 'annual_inc_log', 'application_type', 'avg_cur_bal', 'avg_cur_bal_cap', 'avg_cur_bal_log', 'bc_util', 'bc_util_cap', 'bc_util_corrected', 'bc_util_missing', 'bureau_recovery_flag', 'credit_hist_months', 'defaulted', 'delinq_2yrs', 'dti', 'dti_cap', 'earliest_cr_dt', 'earliest_cr_invalid', 'earliest_cr_line', 'emp_length', 'emp_title', 'ever_delinq', 'fico_range_high', 'fico_range_low', 'fico_swapped', 'grade', 'home_ownership', 'id', 'inq_last_6mths', 'inst_ratio', 'installment', 'installment_mismatch', 'int_rate', 'issue_d', 'issue_dt', 'loan_amnt', 'loan_amnt_corrected', 'mort_acc', 'mths_since_last_delinq', 'num_tl_op_past_12m', 'open_acc', 'pub_rec', 'pub_rec_bankruptcies', 'purpose', 'revol_bal', 'revol_bal_cap', 'revol_bal_log', 'revol_util', 'revol_util_cap', 'revol_util_corrected', 'sub_grade', 'term', 'term_months', 'title', 'tot_cur_bal', 'tot_cur_bal_cap', 't

In [15]:
AUDIT_FLAGS = ["annual_inc_corrected", "bc_util_corrected", "revol_util_corrected",
               "loan_amnt_corrected", "fico_swapped", "earliest_cr_invalid",
               "installment_mismatch", "zip_state_mismatch"]

EXCLUDE = {
    "bureau_recovery_flag": "leakage: recovery happens after charge-off (AUC 0.906; 100% default when 1)",
    "defaulted": "target (y), not an input",
    "grade": "old model output; used as benchmark",
    "sub_grade": "old model output; used as benchmark",
    "int_rate": "old model output (price set from sub_grade); tested in Part B ablation",
    "id": "identifier",
    "issue_d": "raw date text",
    "issue_dt": "used for the time split only",
    "earliest_cr_line": "raw date text",
    "earliest_cr_dt": "replaced by credit_hist_months",
    "title": "redundant with purpose (63% identical, A1)",
    "term": "replaced by term_months",
    "addr_state": "replaced by addr_state_clean",
    "zip_code": "3-digit location: high cardinality and a proxy risk for protected groups",
    "inst_ratio": "audit helper from A1",
}
EXCLUDE.update({f: "audit flag: marks planted errors, absent from holdout" for f in AUDIT_FLAGS})
for c, rule in CAPS.items():                      # raw/intermediate versions of treated fields
    if rule["log1p"]:
        EXCLUDE[c] = f"replaced by {c}_log"
        EXCLUDE[f"{c}_cap"] = f"intermediate; replaced by {c}_log"
    else:
        EXCLUDE[c] = f"replaced by {c}_cap"

PENDING = {
    "emp_title": "group into job families (A5)",
    "addr_state_clean": "location: decide after proxy analysis (A7)",
    "fico_range_low": "combine into fico_mid (A4 part 4)",
    "fico_range_high": "combine into fico_mid (A4 part 4)",
}

FEATURES = [c for c in train_imp.columns if c not in EXCLUDE and c not in PENDING]
print(f"excluded {len(EXCLUDE)}, pending {len(PENDING)}, features {len(FEATURES)}, total {len(train_imp.columns)}")
print(sorted(FEATURES))

excluded 38, pending 4, features 30, total 72
['acc_open_past_24mths', 'annual_inc_log', 'application_type', 'avg_cur_bal_log', 'bc_util_cap', 'bc_util_missing', 'credit_hist_months', 'delinq_2yrs', 'dti_cap', 'emp_length', 'ever_delinq', 'home_ownership', 'inq_last_6mths', 'installment', 'loan_amnt', 'mort_acc', 'mths_since_last_delinq', 'num_tl_op_past_12m', 'open_acc', 'pub_rec', 'pub_rec_bankruptcies', 'purpose', 'revol_bal_log', 'revol_util_cap', 'term_months', 'tot_cur_bal_log', 'tot_hi_cred_lim_log', 'total_acc', 'total_rev_hi_lim_log', 'verification_status']


In [16]:
NEW_RATIOS = ["loan_to_income", "installment_to_income"]
NEW_FEATURES = ["fico_mid"] + NEW_RATIOS + ["revol_bal_zero"]

def add_features(df, feat_caps=None):
    """Derived features; ratios are capped when feat_caps (fitted on ≤ 2014) is given."""
    out = df.copy()
    income = out["annual_inc_cap"]
    out["fico_mid"] = (out["fico_range_low"] + out["fico_range_high"]) / 2
    out["loan_to_income"] = out["loan_amnt"] / income
    out["installment_to_income"] = out["installment"] / (income / 12)
    out["revol_bal_zero"] = (out["revol_bal"] == 0).astype(int)
    if feat_caps:
        for c, upper in feat_caps.items():
            out[c] = out[c].clip(upper=upper)
    return out

# fit the ratio caps on ≤ 2014 only, then apply to all years
tmp = add_features(train_imp[train_imp["issue_dt"] <= FIT_CUTOFF])
FEAT_CAPS = {c: float(tmp[c].quantile(0.995)) for c in NEW_RATIOS}
train_feat = add_features(train_imp, FEAT_CAPS)

# update the column lists
for c in ["fico_range_low", "fico_range_high"]:
    PENDING.pop(c, None)
    EXCLUDE[c] = "replaced by fico_mid"
FEATURES = sorted(set(FEATURES + NEW_FEATURES))
print(FEAT_CAPS, "| features:", len(FEATURES))

fit_feat = train_feat[train_feat["issue_dt"] <= FIT_CUTOFF]
print(fit_feat[NEW_FEATURES].describe().T[["mean", "50%", "max"]].round(3))
single_feature_auc(fit_feat[NEW_FEATURES + ["loan_amnt", "annual_inc_log", "installment", "dti_cap", TARGET]])

{'loan_to_income': 0.46609333333333297, 'installment_to_income': 0.18774666666666665} | features: 34
                          mean      50%      max
fico_mid               695.453  687.000  842.000
loan_to_income           0.221    0.208    0.466
installment_to_income    0.081    0.077    0.188
revol_bal_zero           0.003    0.000    1.000


,column,auc,missing
0,loan_to_income,0.591,0
1,fico_mid,0.586,0
2,installment_to_income,0.575,0
3,dti_cap,0.574,0
4,annual_inc_log,0.552,0
5,loan_amnt,0.538,0
6,installment,0.524,0
7,revol_bal_zero,0.501,0


### A4 Step 4: Derived features

Kept:
- **fico_mid** = mean of the FICO range (the pair is always 4 apart). Same information (AUC 0.586), one column.
- **loan_to_income** = loan_amnt / capped annual income (cap p99.5 = 0.466, fitted on ≤ 2014). AUC 0.591 vs 0.538 (loan_amnt) and 0.552 (income): stronger than either part, and than FICO.

Tested and rejected:
- **installment, installment_to_income**: the installment is fixed by amount, term and int_rate (Spearman 0.9997 / 0.9990 between installment/loan_amnt and int_rate within 36 / 60-month terms), so it carries the old model's pricing. Excluded like int_rate; kept for the Part B ablation.
- **principal_per_year_to_income** (rate-free yearly burden): AUC 0.532, Spearman 0.897 with loan_to_income. Dividing by the term works against the term effect: 60-month loans default more despite a lower yearly burden, so their risk comes from who chooses long terms, not from the yearly amount.
- **revol_bal_zero**: redundant with revol_bal_log (zeros sit at 0) and too rare (0.3%).

Final input list: **31 features**; every one of the 76 columns is classified (43 excluded with reasons, 2 pending).

In [18]:
# --- 1. Evidence: installment carries int_rate (why it's excluded) ---
for t in [36, 60]:
    d = train_feat[(train_feat["issue_dt"] <= FIT_CUTOFF) & (train_feat["term_months"] == t)]
    pay_ratio = d["installment"] / d["loan_amnt"]
    print(f"{t} months  spearman(installment/loan_amnt, int_rate): "
          f"{pay_ratio.corr(d['int_rate'], method='spearman'):.4f}")

# --- 2. Rate-free yearly burden (replaces installment_to_income) ---
def add_yearly_burden(df, cap=None):
    """Principal repaid per year relative to yearly income; no interest rate involved."""
    out = df.copy()
    years = out["term_months"] / 12
    out["principal_per_year_to_income"] = (out["loan_amnt"] / years) / out["annual_inc_cap"]
    if cap is not None:
        out["principal_per_year_to_income"] = out["principal_per_year_to_income"].clip(upper=cap)
    return out

is_fit = train_feat["issue_dt"] <= FIT_CUTOFF
tmp = add_yearly_burden(train_feat[is_fit])
FEAT_CAPS.pop("installment_to_income", None)
FEAT_CAPS["principal_per_year_to_income"] = float(tmp["principal_per_year_to_income"].quantile(0.995))
train_feat = add_yearly_burden(train_feat, FEAT_CAPS["principal_per_year_to_income"])
train_feat = train_feat.drop(columns="revol_bal_zero", errors="ignore")

fit_feat = train_feat[is_fit]
compare = single_feature_auc(fit_feat[["loan_to_income", "principal_per_year_to_income",
                                       "term_months", "installment_to_income", TARGET]])
print(compare)
print("spearman(loan_to_income, principal_per_year_to_income):",
      round(fit_feat["loan_to_income"].corr(fit_feat["principal_per_year_to_income"], method="spearman"), 3))

# --- 3. Final lists ---
EXCLUDE["installment"] = "encodes int_rate given amount and term (old-model pricing); Part B ablation group"
EXCLUDE["installment_to_income"] = "built on installment, so carries int_rate; Part B ablation group"
FEATURES = sorted((set(FEATURES) - {"revol_bal_zero", "installment", "installment_to_income"})
                  | {"principal_per_year_to_income"})
NEW_FEATURES = ["fico_mid", "loan_to_income", "principal_per_year_to_income"]
print("features:", len(FEATURES), "| excluded:", len(EXCLUDE), "| pending:", len(PENDING),
      "| columns:", len(train_feat.columns))

# --- 4. Save the A4 rulebook and data ---
A4_SPEC = {"impute": IMPUTE, "feature_caps": FEAT_CAPS,
           "exclude": EXCLUDE, "pending": PENDING, "features": FEATURES}
with open("../reports/a4_spec.json", "w") as f:
    json.dump(A4_SPEC, f, indent=2)
train_feat.to_parquet("../data/interim/train_a4.parquet", index=False)

36 months  spearman(installment/loan_amnt, int_rate): 0.9997
60 months  spearman(installment/loan_amnt, int_rate): 0.9990
                         column    auc  missing
0                   term_months  0.603        0
1                loan_to_income  0.591        0
2         installment_to_income  0.575        0
3  principal_per_year_to_income  0.532        0
spearman(loan_to_income, principal_per_year_to_income): 0.897
features: 32 | excluded: 42 | pending: 2 | columns: 76


In [19]:
FEATURES = [c for c in FEATURES if c != "principal_per_year_to_income"]
FEAT_CAPS.pop("principal_per_year_to_income", None)
EXCLUDE["principal_per_year_to_income"] = ("tested: weaker than loan_to_income (AUC 0.532 vs 0.591), "
                                           "spearman 0.897 with it; dividing by term opposes the term effect")
NEW_FEATURES = ["fico_mid", "loan_to_income"]
print("features:", len(FEATURES), "| excluded:", len(EXCLUDE), "| pending:", len(PENDING),
      "| columns:", len(train_feat.columns))                    # expect 31 | 43 | 2 | 76

A4_SPEC = {"impute": IMPUTE, "feature_caps": FEAT_CAPS,
           "exclude": EXCLUDE, "pending": PENDING, "features": FEATURES}
with open("../reports/a4_spec.json", "w") as f:
    json.dump(A4_SPEC, f, indent=2)

features: 31 | excluded: 43 | pending: 2 | columns: 76
